# UNI 02 — train attention-MIL head on UNI embeddings



In [ ]:
!pip install -q timm

In [ ]:
import os, numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import cohen_kappa_score, confusion_matrix
import matplotlib.pyplot as plt
from IPython.display import clear_output
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu'); print(device)

In [ ]:
EMB_DIR  = '/kaggle/input/datasets/shashaboii/panda-uni-embeddings1/uni_embeddings'
EMB_DIM  = 1024; N_TILES = 36; N_CLASSES = 6
HEAD_NAME = 'uni_mil_fold0'
EPOCHS = 30; LR = 1e-4; BATCH = 32; SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
df = pd.read_csv(os.path.join(EMB_DIR, '/kaggle/input/competitions/prostate-cancer-grade-assessment/train.csv'))
have = set(f[:-4] for f in os.listdir(EMB_DIR) if f.endswith('.npy'))
df = df[df.image_id.isin(have)].reset_index(drop=True)
print('slides with embeddings:', len(df)); print(df.data_provider.value_counts())

In [ ]:
def ordinal_target(g):
    t = np.zeros(N_CLASSES - 1, np.float32); t[:int(g)] = 1.; return t

class BagDataset(Dataset):
    def __init__(self, frame): self.frame = frame.reset_index(drop=True)
    def __len__(self): return len(self.frame)
    def __getitem__(self, i):
        row = self.frame.iloc[i]
        emb = np.load(os.path.join(EMB_DIR, f'{row.image_id}.npy')).astype(np.float32)
        if emb.shape[0] != N_TILES:
            emb = emb[:N_TILES] if emb.shape[0] > N_TILES else np.pad(emb, [[0, N_TILES-emb.shape[0]],[0,0]])
        return torch.from_numpy(emb), torch.from_numpy(ordinal_target(row.isup_grade)), int(row.isup_grade)

In [ ]:
class GatedAttentionMIL(nn.Module):
    def __init__(self, in_dim=EMB_DIM, hid=256, att=128, out_dim=5, p=0.25):
        super().__init__()
        self.fc   = nn.Sequential(nn.Linear(in_dim, hid), nn.ReLU(), nn.Dropout(p))
        self.attV = nn.Linear(hid, att); self.attU = nn.Linear(hid, att); self.attw = nn.Linear(att, 1)
        self.head = nn.Linear(hid, out_dim)
    def forward(self, x):                                          # x: (B, N, D)
        h = self.fc(x)
        a = torch.tanh(self.attV(h)) * torch.sigmoid(self.attU(h))
        a = torch.softmax(self.attw(a).squeeze(-1), dim=1)
        m = torch.bmm(a.unsqueeze(1), h).squeeze(1)
        return self.head(m), a

In [ ]:
def qwk(t, p): return cohen_kappa_score(t, p, weights='quadratic') if len(p) else float('nan')
def decode(logits): return torch.sigmoid(logits).sum(1).round().clamp(0, N_CLASSES-1)

def run(model, loader, crit, opt=None):
    train = opt is not None; model.train(train)
    L, P, T = [], [], []
    for emb, tgt, g in loader:
        emb, tgt = emb.to(device), tgt.to(device)
        with torch.set_grad_enabled(train):
            logits, _ = model(emb); loss = crit(logits, tgt)
            if train: opt.zero_grad(); loss.backward(); opt.step()
        L.append(loss.item()); P.append(decode(logits).detach().cpu()); T.append(g)
    P = torch.cat(P).numpy(); T = torch.cat(T).numpy()
    return float(np.mean(L)), qwk(T, P), T, P

def live(h):
    clear_output(wait=True); ep = range(1, len(h['tl'])+1)
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    ax[0].plot(ep, h['tl'], '-o', label='train'); ax[0].plot(ep, h['vl'], '-o', label='val')
    ax[0].set_title('BCE loss'); ax[0].legend()
    ax[1].plot(ep, h['vk'], '-o'); ax[1].set_ylim(0,1); ax[1].set_title('val QWK'); plt.tight_layout(); plt.show()

def train_model(tr, va, epochs=EPOCHS, show=True):
    model = GatedAttentionMIL().to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    crit = nn.BCEWithLogitsLoss()
    tl = DataLoader(BagDataset(tr), batch_size=BATCH, shuffle=True)
    vl = DataLoader(BagDataset(va), batch_size=BATCH, shuffle=False)
    h = {'tl': [], 'vl': [], 'vk': []}; best, bs, be = -1, None, None
    for _ in range(epochs):
        trl, _, _, _ = run(model, tl, crit, opt)
        with torch.no_grad(): vll, vk, yt, yp = run(model, vl, crit)
        h['tl'].append(trl); h['vl'].append(vll); h['vk'].append(vk)
        if show: live(h)
        if vk > best: best, bs, be = vk, {k: v.cpu() for k, v in model.state_dict().items()}, (yt, yp)
    model.load_state_dict(bs); return model, best, be

## Main run (stratified split)

In [ ]:
skf = StratifiedKFold(5, shuffle=True, random_state=SEED)
tr_i, va_i = next(skf.split(df, df.isup_grade))
model, best, (yt, yp) = train_model(df.iloc[tr_i], df.iloc[va_i])
torch.save(model.state_dict(), f'/kaggle/working/{HEAD_NAME}.pth')
print('best val QWK:', round(best, 4))

In [ ]:
def plot_cm(t, p, title):
    cm = confusion_matrix(t, p, labels=list(range(N_CLASSES)))
    fig, ax = plt.subplots(figsize=(5, 4)); im = ax.imshow(cm, cmap='Blues')
    for (i, j), v in np.ndenumerate(cm):
        ax.text(j, i, int(v), ha='center', va='center', color='white' if v > cm.max()/2 else 'black')
    ax.set_xticks(range(N_CLASSES)); ax.set_yticks(range(N_CLASSES))
    ax.set_xlabel('predicted'); ax.set_ylabel('true'); ax.set_title(title); plt.colorbar(im); plt.tight_layout(); plt.show()
plot_cm(yt, yp, f'Validation (QWK={best:.3f})')

## Novelty experiment: cross-centre generalisation


In [ ]:
def cross(train_p, test_p):
    tr = df[df.data_provider == train_p]; te = df[df.data_provider == test_p]
    i_tr, i_va = next(StratifiedKFold(5, shuffle=True, random_state=SEED).split(tr, tr.isup_grade))
    m, _, _ = train_model(tr.iloc[i_tr], tr.iloc[i_va], show=False)
    with torch.no_grad():
        _, k, yt, yp = run(m, DataLoader(BagDataset(te), batch_size=BATCH), nn.BCEWithLogitsLoss())
    print(f'train {train_p} -> test {test_p}: QWK={k:.3f} (n={len(te)})')
    return k, yt, yp
k1, t1, p1 = cross('radboud', 'karolinska')
k2, t2, p2 = cross('karolinska', 'radboud')
plot_cm(t1, p1, f'Radboud->Karolinska (QWK={k1:.3f})')
plot_cm(t2, p2, f'Karolinska->Radboud (QWK={k2:.3f})')

val_df_uni = df.iloc[va_i][['image_id','data_provider']].copy()
val_df_uni['true_isup'] = yt; val_df_uni['pred_isup'] = yp
val_df_uni.to_csv('/kaggle/working/val_df_uni.csv', index=False)